# Rebuild `exn_swc_id_map.csv`

Maps each neuron in `excitatory_neurons.csv` to the SWC file holding its skeleton.

The sensory table and the SWC exports were made at different times, so 19 of the 177
neurons carry different root IDs in the two. Matching on the CSV IDs alone finds only
158 skeletons; the missing 19 are then zero-filled and the Figure 2d clustering degrades
from 6 clusters to 4. This notebook resolves the correspondence once and writes it out so
the figure notebook runs offline.

Two rules make it correct:

1. **Search only the seven class folders** — `adltmr_module`, `cltmr_module`,
   `noci_module`, `np_module`, `r`, `v1`, `v2`. Together they hold exactly 177 SWCs, one
   per ExN. `lamian_i_in` is a 413-file superset of lamina I interneurons; including it
   pulls 28 `v1` neurons onto the wrong file.
2. **Try the recorded ID before its descendants.** For two neurons the SWC predates a
   later re-proofreading and exists only under the old ID.

The check at the end must show a clean diagonal: every neuron's SWC in the folder matching
its label. **Requires CAVE.**


In [ ]:
import glob
import os
import re

import caveclient
import numpy as np
import pandas as pd

MATERIALIZATION = 669
CLASS_ORDER = ["adltmr_module", "cltmr_module", "noci_module",
               "np_module", "r", "v1", "v2"]

CSV = "../../data/sensory_input_features/excitatory_neurons.csv"
SWC_ROOT = "../../data/swc_skeletons"
OUT = "../../data/sensory_input_features/exn_swc_id_map.csv"

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

exn = pd.read_csv(CSV)

swc = {}
for cls in CLASS_ORDER:
    for f in glob.glob(os.path.join(SWC_ROOT, cls, "*.swc")):
        m = re.search(r"(\d{15,})", os.path.basename(f))
        if m:
            swc[int(m.group(1))] = (cls, os.path.basename(f))

print(f"{len(exn)} ExNs, {len(swc)} SWCs across the {len(CLASS_ORDER)} class folders")


In [ ]:
rows = []
for old, tag in zip(exn.neuron_id.astype(np.int64), exn.cell_type.str.strip()):
    latest = sorted({int(r) for r in
                     np.asarray(client.chunkedgraph.get_latest_roots([int(old)])).reshape(-1)
                     if r})
    candidates = [old] + [r for r in latest if r != old]   # recorded ID first

    pick = via = None
    for r in candidates:                       # prefer a file whose folder matches the label
        if r in swc and swc[r][0] == tag:
            pick, via = r, ("recorded" if r == old else "descendant")
            break
    if pick is None:
        for r in candidates:
            if r in swc:
                pick, via = r, "label_mismatch"
                break

    rows.append({"csv_root_id": old, "cell_type": tag,
                 "swc_root_id": pick if pick else pd.NA,
                 "swc_folder": swc[pick][0] if pick else pd.NA,
                 "swc_file": swc[pick][1] if pick else pd.NA,
                 "resolved_via": via or "none"})

id_map = pd.DataFrame(rows)
print(id_map.resolved_via.value_counts().to_string())
print(f"\ncoverage: {id_map.swc_root_id.notna().sum()} / {len(id_map)}")
print(f"IDs that differ between the two: "
      f"{(id_map.csv_root_id != id_map.swc_root_id).sum()}")


In [ ]:
# Must be a clean diagonal -- every neuron's SWC in the folder matching its label.
print(pd.crosstab(id_map.cell_type, id_map.swc_folder).to_string())

problems = id_map[id_map.resolved_via.isin(["label_mismatch", "none"])]
if len(problems):
    raise AssertionError(f"{len(problems)} neuron(s) unresolved:\n{problems}")

id_map.to_csv(OUT, index=False)
print(f"\nwrote {OUT}")
